In [ ]:
%run utility_reconciliation

In [ ]:
# Fabric Metadata Config database connection params
server = "<<servername>>.msit-database.fabric.microsoft.com"
database = "<<config databasename>>"

# Business context variables
trackName = "<<track name>>"
databaseName = "<<database name>>"

# Build JDBC URL
config_jdbc_url = f"jdbc:sqlserver://{server};databaseName={database}"
config_token = mssparkutils.credentials.getToken("https://database.windows.net/")

In [ ]:
ddl_statements = [
    # SCHEMAS
    """
    IF NOT EXISTS (
        SELECT 1
        FROM sys.schemas
        WHERE name = 'metadata'
    )
    BEGIN
        EXEC('CREATE SCHEMA [metadata]')
    END
    """,
    """
    IF NOT EXISTS (
        SELECT 1
        FROM sys.schemas
        WHERE name = 'logging'
    )
    BEGIN
        EXEC('CREATE SCHEMA [logging]')
    END
    """,
    # TABLE: metadata.tbl_Reconciliation_ConnectionDetails
    """
    IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'metadata'
        AND t.name = 'tbl_Reconciliation_ConnectionDetails'
    )
    BEGIN
        CREATE TABLE [metadata].[tbl_Reconciliation_ConnectionDetails](
            [ID] [int] IDENTITY(1,1) NOT NULL,
            [TrackName] [nvarchar](500) NULL,
            [DatabaseName] [nvarchar](500) NULL,
            [SourceServerName] [nvarchar](500) NULL,
            [SourceDatabaseName] [nvarchar](500) NULL,
            [TargetServerName] [nvarchar](500) NULL,
            [TargetDatabaseName] [nvarchar](500) NULL,
            [IsActive] [int] NOT NULL,
            [CreatedOn] [datetime] NULL
        ) 

        ALTER TABLE [metadata].[tbl_Reconciliation_ConnectionDetails] ADD  DEFAULT (getdate()) FOR [CreatedOn]
    END
    """,
   # [logging].[tbl_Reconciliation_RecordCountsDiff_Current]
    """IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'logging'
        AND t.name = 'tbl_Reconciliation_RecordCountsDiff_Current'
    )
    BEGIN
        CREATE TABLE [logging].[tbl_Reconciliation_RecordCountsDiff_Current]
        (
            [TABLE_SCHEMA] [nvarchar](max) NULL,
            [TABLE_NAME] [nvarchar](max) NULL,
            [count_prod] [bigint] NULL,
            [count_fabric] [float] NULL,
            [executionDateTime] [datetime] NOT NULL,
            [trackName] [nvarchar](max) NOT NULL,
            [databaseName] [nvarchar](max) NOT NULL
        )
    END""",
    #[logging].[tbl_Reconciliation_SchemaDiff_Current]
    """
        IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'logging'
        AND t.name = 'tbl_Reconciliation_SchemaDiff_Current'
    )
    BEGIN
        CREATE TABLE [logging].[tbl_Reconciliation_SchemaDiff_Current](
            [TABLE_SCHEMA] [nvarchar](max) NULL,
            [TABLE_NAME] [nvarchar](max) NULL,
            [COLUMN_NAME] [nvarchar](max) NULL,
            [DATA_TYPE_Prod] [nvarchar](max) NULL,
            [IS_NULLABLE_Prod] [nvarchar](max) NULL,
            [DATA_TYPE_Fabric] [nvarchar](max) NULL,
            [IS_NULLABLE_Fabric] [nvarchar](max) NULL,
            [_merge] [nvarchar](max) NULL,
            [executionDateTime] [datetime] NOT NULL,
            [trackName] [nvarchar](max) NOT NULL,
            [databaseName] [nvarchar](max) NOT NULL
        ) 
    END
    """, 
    #[logging].[tbl_Reconciliation_TableDiff_Current]
    """
        IF NOT EXISTS (
        SELECT 1
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        WHERE s.name = 'logging'
        AND t.name = 'tbl_Reconciliation_TableDiff_Current'
    )
    BEGIN
        CREATE TABLE [logging].[tbl_Reconciliation_TableDiff_Current](
            [TABLE_SCHEMA] [nvarchar](max) NULL,
            [TABLE_NAME] [nvarchar](max) NULL,
            [COMPARE_RESULT] [nvarchar](max) NULL,
            [databaseName] [nvarchar](max) NULL,
            [trackName] [nvarchar](max) NULL,
            [executionDateTime] [datetime] NULL
        )
    END
    """,
    #========== VIEW ==========
    """
    IF OBJECT_ID('logging.vw_Reconciliation_TableDiff', 'V') IS NOT NULL
        DROP VIEW [logging].[vw_Reconciliation_TableDiff]
    """,
    """
    CREATE VIEW [logging].[vw_Reconciliation_TableDiff]
    AS
    SELECT *
        ,CASE 
            WHEN TABLE_NAME LIKE 'vw%'
                THEN 'VIEW'
            ELSE 'TABLE'
            END AS ObjectType
    FROM logging.tbl_Reconciliation_TableDiff_Current
    """,
    """ 
    IF OBJECT_ID('logging.vw_Reconciliation_SchemaDiff', 'V') IS NOT NULL
        DROP VIEW [logging].[vw_Reconciliation_SchemaDiff]
    """,
    """
    CREATE VIEW [logging].[vw_Reconciliation_SchemaDiff]
    AS
    SELECT TrackName
        ,DatabaseName
        ,TABLE_SCHEMA AS 'SchemaName'
        ,TABLE_NAME AS 'TableName'
        ,COLUMN_NAME AS ColumnName
        ,DATA_TYPE_Prod AS 'DataType(Prod)'
        ,DATA_TYPE_Fabric AS 'DataType(Fabric)'
        ,IS_NULLABLE_Fabric
        ,IS_NULLABLE_Prod
        ,executionDateTime
        ,CASE 
            WHEN DATA_TYPE_Prod <> DATA_TYPE_Fabric
                THEN 'Not Matched'
            ELSE 'Matched'
            END AS Result
        ,CASE 
            WHEN TABLE_NAME LIKE 'vw%'
                THEN 'VIEW'
            ELSE 'TABLE'
            END AS [ObjectType]
    FROM logging.tbl_Reconciliation_SchemaDiff_Current
    """,
    """         
    IF OBJECT_ID('logging.vw_Reconciliation_RecordCountDiff', 'V') IS NOT NULL
        DROP VIEW [logging].[vw_Reconciliation_RecordCountDiff]
    """,
    """
    CREATE VIEW [logging].[vw_Reconciliation_RecordCountDiff]
    AS
    SELECT *
        ,CASE 
            WHEN isnull(count_prod, 0) <> isnull(count_fabric, 0)
                THEN 'Not Matched'
            ELSE 'Matched'
            END AS Result
        ,CASE 
            WHEN count_prod = 0
                AND count_fabric = 0
                THEN 0
            WHEN count_prod = 0
                AND count_fabric > 0
                THEN 100
            ELSE ROUND(((count_fabric - count_prod) * 100.0) / count_prod, 2)
            END AS VariancePercentage
        ,CASE 
            WHEN TABLE_NAME LIKE 'vw%'
                THEN 'VIEW'
            ELSE 'TABLE'
            END AS ObjectType
        ,case 		
            when TABLE_NAME ='<<ObjectName>>' and databaseName='<<DATABASE>>'
            then 'Data is stale in Prod'
            else
            ''
            end as ExceptionScenario,
            FORMAT(executionDateTime AT TIME ZONE 'UTC' AT TIME ZONE 'Pacific Standard Time','yyyy-MM-dd HH:00:00'
    ) AS FilterDate
    FROM logging.tbl_Reconciliation_RecordCountsDiff_Current
    """,
]

In [ ]:
from datetime import datetime
# Execute them
results = execute_sql_statements(ddl_statements, config_jdbc_url, config_token)

# Optionally, check results / log
for res in results:
    print(res)

In [ ]:
# Dictionary for entering Power BI report details into metadata.tbl_LoadTestConnectionDetailsDAX table
connectionDetails = [
    {
        "TrackName": "Track1",
        "DatabaseName": "Database1",
        "SourceServerName": "SourceServer.database.windows.net",
        "SourceDatabaseName": "SourceDB1",
        "TargetServerName": "TargetServer.msit-datawarehouse.fabric.microsoft.com",
        "TargetDatabaseName": "TargetDB1",
        "isActive": True
    },
    {
        "TrackName": "Track2",
        "DatabaseName": "Database2",
        "SourceServerName": "SourceServer.database.windows.net",
        "SourceDatabaseName": "SourceDB2",
        "TargetServerName": "TargetServer.msit-datawarehouse.fabric.microsoft.com",
        "TargetDatabaseName": "TargetDB2",
        "isActive": True
    }
]

In [ ]:
def my_exec_fn(sql_text: str):
    return executeSQLStatementsWithToken([sql_text], config_jdbc_url)

results =  batch_insert_conn_if_not_exists(connectionDetails, config_jdbc_url, my_exec_fn)

In [ ]:
# --- Configuration & Metadata Retrieval ---

# Query to fetch source/target servers & DBs
config_query = f"""
SELECT DISTINCT
    TrackName,
    sourceServerName,
    sourceDatabaseName,
    targetServerName,
    targetDatabaseName
FROM metadata.tbl_Reconciliation_ConnectionDetails
WHERE TrackName = '{trackName}'
  AND DatabaseName = '{databaseName}'
"""

# Execute the query and get a Pandas DataFrame
config_df = read_sql_data(config_query, config_jdbc_url, get_access_token())

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(config_df)

# Validate that we got at least one row
if spark_df.rdd.isEmpty():
    raise ValueError(f"No configuration found for TrackName={trackName}, DatabaseName={databaseName}")

# Extract values from the first row
first_row = spark_df.first()
src_server = first_row["sourceServerName"]
src_db     = first_row["sourceDatabaseName"]
tgt_server = first_row["targetServerName"]
tgt_db     = first_row["targetDatabaseName"]

In [ ]:
# Construct JDBC URLs for source (prod) and target (fabric)
prod_jdbc_url = f"jdbc:sqlserver://{src_server}:1433;database={src_db}"
fabric_jdbc_url = f"jdbc:sqlserver://{tgt_server}:1433;database={tgt_db}"

# Acquire AAD access tokens for SQL connections
prod_token = mssparkutils.credentials.getToken("https://database.windows.net/")
fabric_token = mssparkutils.credentials.getToken("https://database.windows.net/")

# Optional: validate token strings are not empty
if not prod_token or not fabric_token:
    raise RuntimeError("Failed to acquire one or both access tokens for prod/fabric SQL endpoints.")

In [ ]:
if __name__ == "__main__":
    try:
        main(
            prod_jdbc_url,
            prod_token,
            fabric_jdbc_url,
            fabric_token,
            trackName,
            databaseName,
            config_jdbc_url  # if your main expects this too
        )
    except Exception as e:
        print(f"Unexpected error during main execution: {e}")
        # Optionally, log stack trace or re-raise depending on your error handling strategy